# Descarga de ensamblados → DriveVerifica los ensamblados candidatos contra NCBI y baja los confirmados a`tesis/70_genomas/`, sin pasar por tu disco.**Por qué acá:** la sesión de Claude tiene NCBI bloqueado por política, así queno puede ni verificar ni bajar. Colab sí.Toda la lógica vive en `scripts/fetch_genomes.sh` — este notebook solo lomaneja. Corré antes `00_setup.ipynb`.

## Preámbulo: montar Drive y clonar el repoEl repo es público, así que el clon no necesita credenciales. **Los notebooksllaman a los scripts del repo en vez de reimplementarlos**: el criterio deselección de corridas y el de verificación de ensamblados tienen que vivir enun solo lugar, o dejan de ser reproducibles.

In [ ]:
from google.colab import drivedrive.mount('/content/drive')import os, pathlibDRIVE = pathlib.Path('/content/drive/MyDrive/tesis')CLON  = pathlib.Path('/content/tesis')assert DRIVE.exists(), f'no veo {DRIVE} — ¿montaste la cuenta correcta?'print('Drive OK:', DRIVE)

In [ ]:
import subprocessif CLON.exists():    print(subprocess.run(['git','-C',str(CLON),'pull','--ff-only'],                         capture_output=True, text=True).stdout)else:    print(subprocess.run(['git','clone','--depth','1','https://github.com/youkonskernel-afk/tesis.git',str(CLON)],                         capture_output=True, text=True).stderr)print(subprocess.run(['git','-C',str(CLON),'log','--oneline','-1'],                     capture_output=True, text=True).stdout)

In [ ]:
import os, shutil, globSRA_VER = '3.1.1'c = glob.glob(f'/opt/sratoolkit.{SRA_VER}*/bin')if c: os.environ['PATH'] = c[0] + ':' + os.environ['PATH']assert shutil.which('jq'), 'falta jq — corré 00_setup.ipynb'GENOMAS = DRIVE / '70_genomas'GENOMAS.mkdir(parents=True, exist_ok=True)print('destino:', GENOMAS)

## 1. Qué falta`candidato` = propuesto pero **no comprobado**. El script se niega a bajar esoshasta que una persona los verifique. Un ensamblado equivocado no fallaruidosamente: alinea peor y contamina la anotación.

In [ ]:
!cd /content/tesis && ./scripts/fetch_genomes.sh estado

## 2. Verificar contra NCBIPara cada organismo pregunta dos cosas: si el accession propuesto existe, y cuáles el ensamblado de **referencia vigente** de la especie. La segunda importa másque la primera — un accession puede existir y no ser el que corresponde.**Leé la salida antes de seguir.**

In [ ]:
!cd /content/tesis && ./scripts/fetch_genomes.sh resolve

## 3. ConfirmarPoné `verificado` en `data/genomas.tsv` para los que la salida de arribaconfirmó. Editá la lista y corré la celda.Si el vigente **difiere** del candidato, actualizá también el accession: ganaNCBI, no lo que dice el TSV.

In [ ]:
CONFIRMADOS = []          # p.ej. ['prupe', 'gadmo']CORREGIR = {}             # p.ej. {'galga': 'GCF_016699485.2'}import re, pathlibspec = pathlib.Path('/content/tesis') / 'data' / 'genomas.tsv'lineas = spec.read_text().split('\n')for i, ln in enumerate(lineas):    if ln.startswith('#') or '\t' not in ln:        continue    f = ln.split('\t')    if f[0] in CORREGIR:        f[4] = CORREGIR[f[0]]    if f[0] in CONFIRMADOS:        f[5] = 'verificado'        lineas[i] = '\t'.join(f)    elif f[0] in CORREGIR:        lineas[i] = '\t'.join(f)spec.write_text('\n'.join(lineas))print('marcados:', CONFIRMADOS or '(ninguno — no se va a bajar nada)')

## 4. Bajar a Drive

In [ ]:
!cd /content/tesis && GENOMES_DIR=/content/drive/MyDrive/tesis/70_genomas ./scripts/fetch_genomes.sh fetch

## 5. Cerrar el círculo con gitEl clon es efímero: se pierde al cerrar la sesión. Copiá esta salida al repo ycommiteala — **el checksum versionado es lo que deja constancia de qué genoma seusó**, porque el que queda al lado del FASTA en Drive no prueba nada: quienreemplace el genoma reemplaza el checksum con él.

In [ ]:
led = pathlib.Path('/content/tesis') / 'data' / 'genomas.sha256'print('--- data/genomas.sha256 ---')print(led.read_text() if led.exists() else '(vacío: no se bajó nada)')print('--- data/genomas.tsv ---')print(spec.read_text())